## Load Data and Plot Regime Overlay

This cell loads the processed **training** dataset from `data/processed/<ASSET>/train.csv`,
parses the timestamp as the DataFrame index, and filters out rows with no valid regime label
(`regime = -1`). It then plots the closing price coloured by the market regime:

- **Gray** → Sideways (0)
- **Green** → Trending Up (1)
- **Red** → Trending Down (2)

You can choose which horizon’s regime to visualise by setting the `REGIME_COL` variable
(e.g. `"regime_h3"` for a 3‑bar forecast).  
The `start_idx` / `end_idx` parameters select a slice by integer row positions.

In [ ]:
import sys
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from typing import Optional
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score
)

# ----- Resolve project root -----
try:
    project_root
except NameError:
    project_root = Path.cwd().parent   # e.g. for notebooks/ directory

# ----- Configuration -----
ASSET = "BTCUSDT"           # Change to your asset
REGIME_COL = "regime_h6"      # Default regime column (copy of regime_h6)
                           # Use "regime_h3" etc. for other horizons

# ----- Load data -----
data_path = project_root / "data" / "processed" / ASSET / "test.csv"
print(f"Loading data from: {data_path}")

if not data_path.exists():
    raise FileNotFoundError(f"File not found: {data_path}")

df = pd.read_csv(data_path)
if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df.set_index("timestamp", inplace=True)

print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")

# Keep only rows with a valid regime label (0, 1, or 2)
if REGIME_COL not in df.columns:
    raise KeyError(f"Column '{REGIME_COL}' not found in the dataset.")

df_labeled = df[df[REGIME_COL] >= 0].copy()
print(f"Valid labeled points: {len(df_labeled)} "
      f"(discarded {len(df) - len(df_labeled)} unlabeled rows)")

# ----- Plotting function -----
def plot_regime_overlay(
    df: pd.DataFrame,
    start_idx: Optional[int] = None,
    end_idx: Optional[int] = None,
    figsize: tuple = (16, 6),
    regime_colors: dict = None,
    title: str = "Close Price with Regime Overlay",
    regime_col: str = REGIME_COL,
) -> None:
    """
    Plot the close price with coloured segments indicating market regime.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns 'close' and the chosen regime column.
    start_idx, end_idx : int, optional
        Integer row positions for slicing (iloc).
    regime_colors : dict, optional
        Mapping {0: colour, 1: colour, 2: colour}. Default grey/green/red.
    title : str
        Plot title.
    regime_col : str
        Name of the column containing regime labels (0,1,2).
    """
    if regime_colors is None:
        regime_colors = {0: "gray", 1: "green", 2: "red"}

    # Slice the dataframe
    df_subset = df.iloc[start_idx:end_idx].copy() if start_idx or end_idx else df.copy()
    if df_subset.empty:
        print("No data after slicing. Check start/end indices.")
        return

    fig, ax = plt.subplots(figsize=figsize)
    x = df_subset.index
    close = df_subset["close"].values
    regime = df_subset[regime_col].values

    # Draw continuous coloured segments for each regime
    prev_regime = None
    segment_start = 0
    for i in range(len(df_subset)):
        cur = regime[i]
        if prev_regime is None:
            prev_regime, segment_start = cur, i
        elif cur != prev_regime:
            ax.plot(x[segment_start:i], close[segment_start:i],
                    color=regime_colors.get(prev_regime, "blue"), linewidth=1.5)
            prev_regime, segment_start = cur, i
    # Last segment
    if prev_regime is not None:
        ax.plot(x[segment_start:], close[segment_start:],
                color=regime_colors.get(prev_regime, "blue"), linewidth=1.5)

    # Legend
    patches = [
        mpatches.Patch(color=regime_colors[0], label="Sideways (0)"),
        mpatches.Patch(color=regime_colors[1], label="Trend Up (1)"),
        mpatches.Patch(color=regime_colors[2], label="Trend Down (2)"),
    ]
    ax.legend(handles=patches, loc="upper left")
    ax.set_xlabel("Time")
    ax.set_ylabel("Close Price")
    ax.set_title(title)
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

# ----- Example usage -----
# Plot a slice of 400 bars (rows 100 to 500)
plot_regime_overlay(
    df_labeled,
    start_idx=2500,
    end_idx=3000,
    title=f"{ASSET} – {REGIME_COL} (bars 100–500)",
    regime_col=REGIME_COL,
)

# Evaluate Regime Detection Model (TCN)

This notebook loads a trained TCN regime classifier, evaluates it on the **test set**, and creates visualisations:

- Performance metrics (accuracy, balanced accuracy, F1 per class)
- Confusion matrix
- True vs. predicted regime overlay on the price chart

**Asset‑Agnostic** – change the `ASSET` variable to `BTCUSDT`, `ETHUSDT`, `EURUSD`, or `USDJPY`.

In [ ]:
# ----- Resolve project root -----
try:
    project_root
except NameError:
    project_root = Path.cwd()

sys.path.insert(0, str(project_root))

from regime.config import CONFIG, REGIME_LABEL_NAMES, get_feature_columns , get_forecast_horizons
from regime.dataset import RegimeDataset
from regime.tcn_model import MultiHorizonRegimeTCN

print(f"Project root: {project_root}")

## Configuration

Set the asset you want to evaluate and the forecast horizon (must match a trained model).

In [ ]:
ASSET = "BTCUSDT"          # ← change to BTCUSDT, ETHUSDT, USDJPY, etc.
WINDOW = int(CONFIG["model"]["window"])
HORIZON = 6               # Choose a specific horizon (1..6) you want to evaluate

# FORCE the feature columns from the config – these are the 7 used during training
FEATURE_COLS = get_feature_columns()   # this returns 7 columns

TARGET_COLS = [f"regime_h{h}" for h in get_forecast_horizons()]   # all horizons
CLASS_NAMES = [REGIME_LABEL_NAMES[i] for i in range(3)]

print(f"Asset       : {ASSET}")
print(f"Window      : {WINDOW}")
print(f"Features    : {len(FEATURE_COLS)} -> {FEATURE_COLS}")
print(f"Target cols : {TARGET_COLS}")
print(f"Horizon     : {HORIZON}")

## 1. Load the processed test data

The test CSV is located at `data/processed/<ASSET>/test.csv`.  
We load it, parse the timestamp (if present), and keep only rows with valid regime labels (0,1,2).  
Rows with `regime = -1` are unlabeled and are discarded for evaluation.

In [ ]:
data_path = project_root / "data" / "processed" / ASSET / "test.csv"
if not data_path.exists():
    raise FileNotFoundError(f"Test file not found: {data_path}")

df_test = pd.read_csv(data_path)
if "timestamp" in df_test.columns:
    df_test["timestamp"] = pd.to_datetime(df_test["timestamp"])
    df_test.set_index("timestamp", inplace=True)

print(f"Full test set: {df_test.shape}")

## 2. Build windowed dataset

The `RegimeDataset` class creates fixed‑length windows of features and extracts the corresponding target labels.

In [ ]:
dataset = RegimeDataset(
    df=df_test,
    feature_cols=FEATURE_COLS,      # <- 7 columns from config
    target_cols=TARGET_COLS,      
    window=WINDOW,
)

X_test, y_test, idx_test = dataset.to_numpy()
print(f"Windowed test samples: {len(y_test)}")
print(f"X_test shape: {X_test.shape}")   # should be (N, WINDOW, 7)

## 3. Load the trained TCN model

Checkpoints are saved in `checkpoints/regime/` with the naming pattern `<ASSET>_tcn_multi.pth`.

In [ ]:
checkpoint_path = project_root / "checkpoints" / "regime" / f"{ASSET}_tcn_multi.pth"
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Model checkpoint not found: {checkpoint_path}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Rebuild model with the same architecture as training
tcn_cfg = CONFIG["tcn"]
model = MultiHorizonRegimeTCN(
    input_size=len(FEATURE_COLS),
    num_horizons=len(TARGET_COLS),
    num_channels=tcn_cfg["num_channels"],
    kernel_size=tcn_cfg["kernel_size"],
    dropout=tcn_cfg["dropout"],
    attention_heads=tcn_cfg["attention_heads"],
    attention_dropout=tcn_cfg["attention_dropout"],
    se_reduction=tcn_cfg["se_reduction"],
    stochastic_depth_prob=0.0,           # inference only
    max_seq_len=WINDOW + 50,
).to(device)

model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()
print(f"Loaded model from {checkpoint_path}")

## 4. Evaluate the model

Generate predictions, compute metrics, and display a detailed classification report.

In [ ]:
# Model expects input shape (batch, seq_len, features) – no permutation needed
X_tensor = torch.from_numpy(X_test).to(device)  # (N, WINDOW, 7)

with torch.no_grad():
    logits = model(X_tensor)                 # (N, num_horizons, 3)
    proba_all = torch.softmax(logits, dim=-1).cpu().numpy()  # (N, H, 3)

# Select the horizon index
horizon_idx = get_forecast_horizons().index(HORIZON)
proba = proba_all[:, horizon_idx, :]          # (N, 3)
y_pred = proba.argmax(axis=1)                # (N,)
y_true = y_test[:, horizon_idx]              # ground truth for this horizon

# Filter out invalid samples (where y_true == -1)
valid_mask = y_true != -1
y_true_valid = y_true[valid_mask]
y_pred_valid = y_pred[valid_mask]

print(f"Valid samples for horizon h={HORIZON}: {len(y_true_valid)}")

# Compute metrics
acc = accuracy_score(y_true_valid, y_pred_valid)
bal_acc = balanced_accuracy_score(y_true_valid, y_pred_valid)
f1_macro = f1_score(y_true_valid, y_pred_valid, average='macro')

print(f"\nAccuracy: {acc:.4f}")
print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1 Macro: {f1_macro:.4f}")
print("\nClassification Report:")
print(classification_report(y_true_valid, y_pred_valid, target_names=CLASS_NAMES, labels=[0,1,2]))

## 5. Confusion matrix

A heatmap shows how well the model distinguishes between Sideways, Trending Up, and Trending Down.

In [ ]:
cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[0, 1, 2])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix – {ASSET} (h={HORIZON})")
plt.tight_layout()
plt.show()

## 6. Visual comparison: true vs. predicted regimes

For a chosen slice of the test set, the top subplot shows the closing price coloured by the **true** regime, and the bottom subplot shows it coloured by the **predicted** regime.

In [ ]:
def plot_regime_comparison(
    df: pd.DataFrame,
    dataset: RegimeDataset,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: list,
    start_idx: int = 0,
    length: int = 300,
    figsize: tuple = (16, 8),
):
    """
    Two subplots: close price coloured by true regime (top) and predicted regime (bottom).
    """
    positions = dataset.indices   # corrected attribute name
    if len(positions) == 0:
        print("No samples to plot.")
        return

    end_idx = min(start_idx + length, len(positions))
    slice_pos = positions[start_idx:end_idx]
    slice_true = y_true[start_idx:end_idx]
    slice_pred = y_pred[start_idx:end_idx]

    # Get the corresponding rows from the original dataframe
    df_slice = df.iloc[slice_pos].copy()
    df_slice["true_regime"] = slice_true
    df_slice["pred_regime"] = slice_pred

    regime_colors = {0: "gray", 1: "green", 2: "red"}

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, sharex=True)

    # True regime
    for regime in [0, 1, 2]:
        mask = df_slice["true_regime"] == regime
        ax1.scatter(df_slice.index[mask], df_slice["close"][mask],
                    color=regime_colors[regime], s=10, label=class_names[regime], alpha=0.8)
    ax1.plot(df_slice.index, df_slice["close"], color="black", alpha=0.3, linewidth=0.8)
    ax1.set_ylabel("Close Price")
    ax1.set_title("True Regimes")
    ax1.legend(loc="upper left")
    ax1.grid(True, alpha=0.3)

    # Predicted regime
    for regime in [0, 1, 2]:
        mask = df_slice["pred_regime"] == regime
        ax2.scatter(df_slice.index[mask], df_slice["close"][mask],
                    color=regime_colors[regime], s=10, alpha=0.8)
    ax2.plot(df_slice.index, df_slice["close"], color="black", alpha=0.3, linewidth=0.8)
    ax2.set_ylabel("Close Price")
    ax2.set_xlabel("Timestamp")
    ax2.set_title("Predicted Regimes")
    ax2.grid(True, alpha=0.3)

    fig.suptitle(f"{ASSET} – True vs Predicted Regimes (h={HORIZON})", fontsize=14)
    plt.tight_layout()
    plt.show()

# Choose a slice (e.g., 300 points starting at the 500th valid sample)
plot_regime_comparison(
    df=df_test,                     # use the full test dataframe (with close price)
    dataset=dataset,
    y_true=y_true_valid,
    y_pred=y_pred_valid,
    class_names=CLASS_NAMES,
    start_idx=500,
    length=300,
)